# Off the Prior: cheap fixes (full fine-tune vs LoRA vs data mixing) on Colab GPU

Fine-tunes the four Timestral sizes (5M, 28M, 123M, 209M) on failure families and scores them against the
pretrained model and the true-dynamics oracle.

- Methods: `full` (all parameters), `lora` (rank-r adapters on the attention and MLP linears), each with replay
  shares `0` (no mixing) and the values in `REPLAY_FRACS` (a share of every batch is in-prior replay series).
- Protocols: `lofo` (leave one family out: train on three of chaos / regime / tails / mixture, test on the fourth)
  and `knob` (train on one range of each family's knob, test on the other range). Splits: `otp/finetune_splits.py`.
- Regression check: fresh in-prior series (and the real-data windows when `bench/regression/real.npz` is in the repo).

Everything this notebook produces goes to `results/finetune/colab/<RUN_ID>/` (small files: scores, summaries,
provenance; zipped for download at the end). Checkpoints go to `checkpoints/finetune/colab/<RUN_ID>/` (large; not
for git). Set `TINY = True` for a smoke test with tiny budgets (runs on CPU too).

Run the cells top to bottom. Set the GPU runtime first (Runtime > Change runtime type).

In [ ]:
# ---- Config cell: every budget and switch lives here -------------------------------------------------
TINY = False                      # True: smoke test (one size, 2 steps, 3 eval samples)
RUN_ID = "run1"                   # results go to results/finetune/colab/<RUN_ID>/
SKIP_INSTALL = False              # True when the dependencies are already installed

SIZES = ["5M", "28M", "123M", "209M"]
PROTOCOLS = ["lofo", "knob"]
HOLDOUTS = ["chaos", "regime", "tails", "mixture"]
METHODS = ["full", "lora"]
REPLAY_FRACS = [0.0, 0.3, 0.5]    # 0.0 = no mixing
# Budgets per size: optimizer steps, sequences per step, learning rates (full / LoRA), LoRA rank and alpha.
STEPS = {"5M": 500, "28M": 500, "123M": 300, "209M": 300}
BATCH = {"5M": 32, "28M": 32, "123M": 16, "209M": 16}
LR_FULL = {"5M": 5e-5, "28M": 3e-5, "123M": 2e-5, "209M": 1e-5}
LR_LORA = {"5M": 1e-3, "28M": 1e-3, "123M": 5e-4, "209M": 5e-4}
RANK, ALPHA = 8, 16.0
POOL_BATCHES = 500                # batches of simulated training series cached per family (shared by all sizes)
POOL_WORKERS = 8                  # processes that simulate training series
CONTEXTS = [256, 512, 1024, 2048]
HORIZONS = [32, 128, 512]
N_EVAL = 32                       # held-out samples per family and protocol (500 oracle paths each)
N_REGRESSION = 300                # fresh in-prior series for the regression check
N_PATHS = 500

REPO_URL = ""                     # e.g. "https://github.com/<user>/off-the-prior.git"; or upload off-the-prior.zip

if TINY:
    SIZES, PROTOCOLS, HOLDOUTS = ["5M"], ["lofo"], ["regime", "tails"]
    METHODS, REPLAY_FRACS = ["full", "lora"], [0.0, 0.3]
    STEPS, BATCH, POOL_BATCHES, POOL_WORKERS = {"5M": 2}, {"5M": 4}, 2, 2
    CONTEXTS, HORIZONS = [256], [32]
    N_EVAL, N_REGRESSION, N_PATHS = 3, 3, 20
    RUN_ID = RUN_ID + "_tiny"

RESULTS = f"results/finetune/colab/{RUN_ID}"
CKPTS = f"checkpoints/finetune/colab/{RUN_ID}"
POOLS = "checkpoints/finetune/colab/pools" + ("_tiny" if TINY else "")

## 1. Get the repo (clone, or unzip an uploaded `off-the-prior.zip`)

In [ ]:
import os, subprocess, sys, time, zipfile

def sh(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, check=False)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")
    return r.returncode

# Option A: clone
if REPO_URL and not os.path.exists("off-the-prior"):
    sh(f"git clone --depth 1 {REPO_URL} off-the-prior")
# Option B: unzip an uploaded archive (made with `git archive --format=zip -o off-the-prior.zip HEAD`)
if not os.path.exists("off-the-prior") and os.path.exists("off-the-prior.zip"):
    with zipfile.ZipFile("off-the-prior.zip") as z:
        z.extractall("off-the-prior")
assert os.path.exists("off-the-prior"), "set REPO_URL or upload off-the-prior.zip to the Colab file pane"
os.chdir("off-the-prior")
sys.path.insert(0, os.getcwd())  # so `import otp` works without installing
print(os.getcwd())

## 2. Install and pick the device

In [ ]:
if not SKIP_INSTALL:
    sh('pip -q install "git+https://github.com/mistralai/timestral.git" statsmodels safetensors')
    sh("pip -q install -e . --no-deps")
import torch
os.environ["OTP_DEVICE"] = "cuda" if torch.cuda.is_available() else "cpu"
os.environ["OTP_THREADS"] = str(max(1, (os.cpu_count() or 2) // 2))
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("torch", torch.__version__, "device", os.environ["OTP_DEVICE"], GPU, "cpus", os.cpu_count())

## 3. Train every run

One subprocess per run (so each run reports its own peak memory); runs whose `run.json` exists are skipped, so the cell
can be re-run after a disconnect. Training series are simulated once per family into `POOLS` and shared by all runs
(a paired comparison: every method sees the same batches).

In [ ]:
t_start = time.time()
for size in SIZES:
    cmd = (f"python scripts/finetune.py suite --sizes {size} --protocols {' '.join(PROTOCOLS)} "
           f"--holdouts {' '.join(HOLDOUTS)} --methods {' '.join(METHODS)} "
           f"--replay-fracs {' '.join(map(str, REPLAY_FRACS))} --steps {STEPS[size]} --batch-size {BATCH[size]} "
           f"--lr-full {LR_FULL[size]} --lr-lora {LR_LORA[size]} --rank {RANK} --alpha {ALPHA} "
           f"--pool-batches {POOL_BATCHES} --pool-workers {POOL_WORKERS} --warmup {max(1, STEPS[size] // 20)} "
           f"--contexts {' '.join(map(str, CONTEXTS))} --horizons {' '.join(map(str, HORIZONS))} "
           f"--runs-root {RESULTS}/runs --ckpt-root {CKPTS} --pool-dir {POOLS}")
    sh(cmd)
train_wall = time.time() - t_start
print("training cell wall seconds", round(train_wall, 1))

## 4. Score against the pretrained model, the oracle and (for the regime family) the context-truncation baseline

Writes `scores.csv`, `summary.csv`, `costs.csv`, `provenance.json` and `README.md` into the results folder.
If `bench/regression/real.npz` is in the repo the real-data regression windows are scored too.

In [ ]:
t_eval = time.time()
real = "bench/regression/real.npz"
real_arg = f"--real-npz {real}" if os.path.exists(real) else ""
sh(f"python scripts/eval_finetuned.py --runs-root {RESULTS}/runs --ckpt-root {CKPTS} --out {RESULTS} "
   f"--cache-dir {POOLS}/eval_cache --n-eval {N_EVAL} --n-regression {N_REGRESSION} --n-paths {N_PATHS} {real_arg}")
eval_wall = time.time() - t_eval
from datetime import datetime, timezone
meta = {"run_id": RUN_ID, "gpu": GPU, "device": os.environ["OTP_DEVICE"], "torch": torch.__version__, "tiny": TINY,
        "train_cell_wall_sec": round(train_wall, 1), "eval_cell_wall_sec": round(eval_wall, 1),
        "finished_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sizes": SIZES, "protocols": PROTOCOLS, "holdouts": HOLDOUTS, "methods": METHODS, "replay_fracs": REPLAY_FRACS,
        "steps": STEPS, "batch": BATCH, "lr_full": LR_FULL, "lr_lora": LR_LORA, "rank": RANK, "alpha": ALPHA,
        "pool_batches": POOL_BATCHES, "contexts": CONTEXTS, "horizons": HORIZONS, "n_eval": N_EVAL,
        "n_regression": N_REGRESSION, "n_paths": N_PATHS, "real_regression_set": bool(real_arg)}
from otp.io import write_json
write_json(f"{RESULTS}/colab_meta.json", meta)
print(open(f"{RESULTS}/README.md").read()[:3000])

## 5. Zip the results (no checkpoints) for download; drop the folder into `results/finetune/colab/` in the repo and commit

In [ ]:
# the cached held-out samples are large and regenerate deterministically: leave them out of the archive
sh(f"cd results/finetune/colab && zip -rq ../../../finetune_colab_{RUN_ID}.zip {RUN_ID} -x '{RUN_ID}/eval_cache/*'")
print(os.path.getsize(f"finetune_colab_{RUN_ID}.zip"), "bytes")
try:
    from google.colab import files
    files.download(f"finetune_colab_{RUN_ID}.zip")
except Exception:
    print("no Colab kernel: the archive is at", os.path.abspath(f"finetune_colab_{RUN_ID}.zip"))

## 6. Optional: push checkpoints to the Hugging Face Hub as private repos

Left commented out. Each checkpoint folder `checkpoints/finetune/colab/<RUN_ID>/<run_id>` would go to the private repo
`cuuper22/otp-timestral-<size>-<protocol>-<holdout>-<method>-<nomix|mix<frac>>`, for example
`cuuper22/otp-timestral-28M-lofo-chaos-lora-mix0.3`. Log in first (`huggingface_hub.login()`).
Model-card drafts for these checkpoints are in `results/finetune/model_cards/colab/`.

In [ ]:
# from huggingface_hub import HfApi
# api = HfApi()
# for rid in sorted(os.listdir(CKPTS)):
#     size, ph, method, mix = rid.split("__")
#     protocol, holdout = ph.split("-", 1)
#     repo = f"cuuper22/otp-timestral-{size}-{protocol}-{holdout}-{method}-{mix}"
#     api.create_repo(repo, private=True, exist_ok=True)
#     api.upload_folder(folder_path=f"{CKPTS}/{rid}", repo_id=repo)
#     print("pushed", repo)